# PULSEY Functionality Demo

This notebook demonstrates the main pieces of the PULSEY workflow: importing the package, constructing a pulsating star, computing map coefficients, computing fluxes, sampling surface fluxes, plotting a Mollweide map, creating an animation, and inserting the star into a binary system.

## 0. Imports

The tutorial notebooks import `star` directly from `PULSEY`. We will follow that same style here.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Allow this notebook to be run from either the repository root or examples/.
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "PULSEY").is_dir() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from PULSEY import star

%matplotlib inline

## I. Define Pulsation Modes

A PULSEY `star` is initialized with spherical harmonic mode indices `(l, m)`, plus one frequency, amplitude, and phase for each mode.

In [ ]:
# Each entry in lmArray is one spherical harmonic mode [l, m].
lmArray = [[2, 0], [2, 1], [3, -2]]

# Frequencies, amplitudes, and phases correspond one-to-one with lmArray.
freq = [1.0, 2.5, 0.6]
amp = [0.05, 0.02, 0.015]
phase = [0.0, 0.25, 0.5]

# Additional star configuration.
inc = 75.0
lMax = 3
observedFlag = True

# Times used throughout this notebook.
timeArray = np.linspace(0.0, 5.0, 400)

## II. Initialize A `star` Object

The `star` object stores the pulsation modes and constructs the STARRY-compatible surface map.

In [ ]:
# Initialize the pulsating star.
pulseStar = star(
    lmArray,
    freq,
    amp,
    phase,
    inc=inc,
    lMax=lMax,
    observed=observedFlag,
)

# Show a snapshot of the surface at one time.
pulseStar.show(time=0.25)

## III. Compute Surface Map Coefficients

`computeMap()` returns the dense spherical harmonic coefficient vector used by the underlying STARRY map at each time step.

In [ ]:
# Compute the map coefficient vector for each time.
coeffArray = pulseStar.computeMap(timeArray)

print("Coefficient array shape:", np.asarray(coeffArray).shape)
print("First coefficient vector:")
print(np.asarray(coeffArray[0]))

## IV. Compute A Disk-Integrated Light Curve

`computeFlux()` integrates the visible stellar surface and returns a flux value for each time step.

In [ ]:
# Compute the disk-integrated flux over time.
flux = pulseStar.computeFlux(timeArray)

# The package also has a convenience plot method.
pulseStar.plot(timeArray, flux)

In [ ]:
# The same flux can also be plotted directly with matplotlib.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(timeArray, np.asarray(flux), color="black", lw=1.5)
ax.set_xlabel("Time")
ax.set_ylabel("Disk-integrated flux")
ax.set_title("PULSEY light curve")
ax.grid(alpha=0.25)
plt.show()

## V. Sample Surface Fluxes At Specific Points

`discretizeSurface()` samples local surface flux values at longitude and latitude coordinates. This is a map sample, not a disk-integrated light curve.

In [ ]:
# With grid=False, longitude and latitude arrays are paired point-by-point.
sampleLon, sampleLat, sampleFlux = pulseStar.discretizeSurface(
    time=0.25,
    lon=[0.0, 90.0, 180.0],
    lat=[0.0, 45.0, -30.0],
    grid=False,
)

for lonValue, latValue, fluxValue in zip(sampleLon, sampleLat, sampleFlux):
    print(f"lon={float(lonValue):6.1f} deg, lat={float(latValue):6.1f} deg -> flux={float(fluxValue):.6f}")

## VI. Discretize A Full Surface Grid

If longitude and latitude arrays are omitted, `discretizeSurface()` creates a full latitude-longitude grid with `nLon` by `nLat` samples.

In [ ]:
# Generate a regularly sampled surface grid.
lonGrid, latGrid, fluxGrid = pulseStar.discretizeSurface(
    time=0.25,
    nLon=181,
    nLat=91,
)

print("lon grid shape:", np.asarray(lonGrid).shape)
print("lat grid shape:", np.asarray(latGrid).shape)
print("flux grid shape:", np.asarray(fluxGrid).shape)
print("flux range:", float(np.nanmin(fluxGrid)), "to", float(np.nanmax(fluxGrid)))

## VII. Plot The Surface On A Mollweide Projection

Matplotlib expects Mollweide longitude and latitude coordinates in radians. The reversed seismic colormap is called `seismic_r`.

In [ ]:
# Convert the sampled longitude and latitude grids from degrees to radians.
lonRad = np.deg2rad(np.asarray(lonGrid))
latRad = np.deg2rad(np.asarray(latGrid))

fig = plt.figure(figsize=(9, 4.8))
ax = fig.add_subplot(111, projection="mollweide")

mesh = ax.pcolormesh(
    lonRad,
    latRad,
    np.asarray(fluxGrid),
    cmap="seismic_r",
    shading="auto",
)

ax.grid(True, alpha=0.35)
ax.set_title("PULSEY surface flux at t = 0.25")

cbar = fig.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.08)
cbar.set_label("local surface flux")

plt.show()

## VIII. Apply A Surface Transform Function

`setTransFcn()` stores a transform that is applied to values returned by `discretizeSurface()`. Here we subtract the uniform-map value so the sampled fluxes are expressed as relative surface deviations.

In [ ]:
# Apply a simple transform to surface samples.
pulseStar.setTransFcn(lambda values: values - (1.0 / np.pi))

_, _, deltaFluxGrid = pulseStar.discretizeSurface(
    time=0.25,
    nLon=181,
    nLat=91,
)

# Reset the transform so later cells return the original local flux values.
pulseStar.setTransFcn(None)

print("delta-flux range:", float(np.nanmin(deltaFluxGrid)), "to", float(np.nanmax(deltaFluxGrid)))

## IX. Animate The Pulsation

`Animate()` renders a time sequence, saves `Pulsation.gif`, and displays an HTML animation in notebook environments. This can take a little longer than the static examples above.

In [ ]:
# Keep the demo animation short so it is reasonable to run interactively.
animTime = np.linspace(0.0, 1.0, 30)

# Uncomment this line when you want to render the animation.
# pulseStar.Animate(animTime)

## X. Insert The Star Into A Binary System

`insertBinary()` places the pulsating star into a simple eclipsing binary configuration. `computeBinary()` evaluates the resulting binary light curve.

In [ ]:
# Create a separate star for the binary example.
binaryStar = star(
    [[2, 0]],
    [1.0],
    [0.02],
    [0.0],
    inc=90.0,
    lMax=2,
    observed=True,
)

# Insert a smaller secondary body into the system.
binaryStar.insertBinary(
    m1=1.0,
    r1=1.0,
    m2=0.8,
    r2=0.35,
    period=2.0,
    tTransit=0.25,
)

# Compute and plot the binary flux curve.
binaryTime = np.linspace(0.0, 4.0, 300)
binaryFlux = binaryStar.computeBinary(binaryTime)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(binaryTime, np.asarray(binaryFlux), color="tab:blue", lw=1.5)
ax.set_xlabel("Time")
ax.set_ylabel("Binary flux")
ax.set_title("PULSEY binary light curve")
ax.grid(alpha=0.25)
plt.show()

## XI. Summary

This notebook covered the core PULSEY functions used in the current API: `star`, `show`, `computeMap`, `computeFlux`, `plot`, `discretizeSurface`, `setTransFcn`, `Animate`, `insertBinary`, and `computeBinary`.